In [2]:
import pandas as pd
df = pd.read_parquet('../data/retail_preprocessed.parquet')

In [3]:
import pandas as pd
import numpy as np

df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

if 'Sales' not in df.columns:
    df['Sales'] = df['Quantity'] * df['Price']

data = df[df['Customer ID'].notna()].copy()

# 전체 데이터에 존재하는 모든 고객
all_customers = pd.Index(
    data['Customer ID'].drop_duplicates(),
    name='Customer ID'
)

periods = {
    '2011-09-01 ~ 2011-10-20': ('2011-09-01', '2011-10-21'),
    '2011-10-21 ~ 2011-12-09': ('2011-10-21', '2011-12-10')
}

results = []

for period_name, (start, end) in periods.items():

    period_sales = (
        data.loc[
            (data['InvoiceDate'] >= start) &
            (data['InvoiceDate'] < end)
        ]
        .groupby('Customer ID')['Sales']
        .sum()
        # 해당 기간에 거래가 없는 전체 고객은 0
        .reindex(all_customers, fill_value=0)
    )

    zero_mask = np.isclose(period_sales, 0)

    results.append({
        '기간': period_name,
        '전체 고객 수': len(period_sales),
        'Sales 0 고객 수': zero_mask.sum(),
        'Sales 0 비율(%)': zero_mask.mean() * 100
    })

result_df = pd.DataFrame(results)

print('=== 전체 고객 기준 기간별 Sales 0 비율 ===')
print(result_df.round(2))

=== 전체 고객 기준 기간별 Sales 0 비율 ===
                        기간  전체 고객 수  Sales 0 고객 수  Sales 0 비율(%)
0  2011-09-01 ~ 2011-10-20     5875          3970          67.57
1  2011-10-21 ~ 2011-12-09     5875          3709          63.13


In [ ]:
# 4분위수 / 분산 확인
results = []

for period_name, (start, end) in periods.items():

    customer_sales = (
        data.loc[
            (data['InvoiceDate'] >= start) &
            (data['InvoiceDate'] < end)
        ]
        .groupby('Customer ID')['Sales']
        .sum()
        .reindex(all_customers, fill_value=0)
    )

    stats = customer_sales.describe()
    stats['variance'] = customer_sales.var(ddof=0)
    stats.name = period_name

    results.append(stats)

result_df = pd.DataFrame(results)

print(result_df.round(2))

                          count    mean      std     min  25%  50%     75%  \
2011-09-01 ~ 2011-10-20  5875.0  266.33  1769.87 -561.60  0.0  0.0  214.75   
2011-10-21 ~ 2011-12-09  5875.0  301.12  1522.25 -468.32  0.0  0.0  264.64   

                              max    variance  
2011-09-01 ~ 2011-10-20  72609.63  3131894.93  
2011-10-21 ~ 2011-12-09  56510.44  2316862.09  
